# 02 Architecture Zoo — Convolutional Neural Network (CNN)

**Status:** Complete

**Learning outcome:** Understand why spatial structure matters, implement convolution from scratch in NumPy, build and train a LeNet-style CNN on CIFAR-10, visualise learned filters, and demonstrate that CNNs beat MLPs on image data with fewer parameters.

## Why This Architecture?

An MLP treats every pixel as an independent feature. It flattens a 32x32x3 image into a 3,072-element vector and connects every input to every hidden neuron. This has two fatal problems for images:

1. **No spatial awareness.** The MLP does not know that pixel (0, 0) is adjacent to pixel (0, 1). It would produce exactly the same output if you randomly shuffled all the pixels (as long as training and test data are shuffled the same way). Spatial patterns — edges, textures, shapes — are invisible to it.

2. **Parameter explosion.** A single fully connected hidden layer of 256 neurons on a 32x32x3 image requires 32 x 32 x 3 x 256 = 786,432 weights — just for one layer. Scale to ImageNet (224x224x3) and a single hidden layer of 4,096 neurons would need 600+ million weights.

Convolutional Neural Networks solve both problems with three ideas:

- **Local receptive fields:** Each neuron looks at a small patch of the input (e.g., 3x3 pixels), not the entire image. This encodes the prior that nearby pixels are more related than distant ones.
- **Parameter sharing:** The same small filter (kernel) slides across the entire image. One 3x3x3 filter has only 27 weights, regardless of image size. This drastically reduces parameters and encodes the prior that a useful feature (e.g., a vertical edge) is useful everywhere in the image.
- **Hierarchical composition:** Stacking convolutional layers lets the network build up from edges to textures to parts to objects — each layer's receptive field grows, capturing increasingly abstract features.

## Theory Sketch

### Convolution as a Sliding Dot Product

A 2D convolution slides a small kernel $\mathbf{K}$ (e.g., 3x3) across an input image $\mathbf{I}$, computing a dot product at each position:

$$(\mathbf{I} * \mathbf{K})[i, j] = \sum_{m} \sum_{n} \mathbf{I}[i+m, j+n] \cdot \mathbf{K}[m, n]$$

The output is called a **feature map**. Each position in the feature map tells us "how much does this local patch look like the pattern encoded in the kernel?" A horizontal-edge kernel will produce high activations wherever there is a horizontal edge, regardless of where in the image it appears.

### Feature Maps and Channels

A colour image has 3 input channels (R, G, B). A convolutional layer with $C_{out}$ filters produces $C_{out}$ feature maps. Each filter has shape $(C_{in}, k_H, k_W)$ — it looks at all input channels simultaneously. The total parameters for one conv layer: $C_{out} \times C_{in} \times k_H \times k_W + C_{out}$ (bias).

### Pooling

**Max pooling** takes the maximum value in each non-overlapping patch (e.g., 2x2), reducing spatial dimensions by half. This provides:
- **Spatial downsampling:** reduces computation in subsequent layers
- **Local translation invariance:** small shifts in the input produce the same pooled output
- **Larger effective receptive field:** each neuron in the next layer "sees" a larger portion of the original image

### Output Dimensions

For input size $W$, kernel size $k$, padding $p$, and stride $s$:

$$W_{out} = \left\lfloor \frac{W - k + 2p}{s} \right\rfloor + 1$$

With $k=3, p=1, s=1$: output size equals input size ("same" convolution).
With $k=2, p=0, s=2$ (typical max pool): output size is halved.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time

np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print("Libraries loaded.")

Device: cpu
Libraries loaded.


## From-Scratch Implementation (NumPy)

A simple 2D convolution on a single-channel image with a single filter. This is the core operation — slide a kernel across the image, compute the dot product at each position.

In [2]:
def conv2d_numpy(image, kernel):
    """
    2D convolution (valid mode) on a single-channel image with a single kernel.

    Parameters
    ----------
    image : np.ndarray, shape (H, W)
    kernel : np.ndarray, shape (kH, kW)

    Returns
    -------
    output : np.ndarray, shape (H - kH + 1, W - kW + 1)
    """
    H, W = image.shape
    kH, kW = kernel.shape
    out_H = H - kH + 1
    out_W = W - kW + 1
    output = np.zeros((out_H, out_W), dtype=np.float64)

    for i in range(out_H):
        for j in range(out_W):
            patch = image[i:i+kH, j:j+kW]
            output[i, j] = np.sum(patch * kernel)

    return output


# Create a simple test image: 8x8 with a vertical edge
test_image = np.zeros((8, 8))
test_image[:, 4:] = 1.0  # right half is white

# Define some classic edge detection kernels
kernels = {
    'Vertical edge': np.array([[-1, 0, 1],
                                [-2, 0, 2],
                                [-1, 0, 1]], dtype=np.float64),   # Sobel-x
    'Horizontal edge': np.array([[-1, -2, -1],
                                  [ 0,  0,  0],
                                  [ 1,  2,  1]], dtype=np.float64),  # Sobel-y
    'Sharpen': np.array([[ 0, -1,  0],
                          [-1,  5, -1],
                          [ 0, -1,  0]], dtype=np.float64),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
axes[0].imshow(test_image, cmap='gray')
axes[0].set_title('Input (8x8)\nvertical edge at col 4')

for ax, (name, kernel) in zip(axes[1:], kernels.items()):
    result = conv2d_numpy(test_image, kernel)
    ax.imshow(result, cmap='RdBu_r', vmin=-result.max(), vmax=result.max())
    ax.set_title(f'{name}\noutput shape: {result.shape}')

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('NumPy Convolution From Scratch', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Verify against scipy
from scipy.signal import correlate2d
for name, kernel in kernels.items():
    ours = conv2d_numpy(test_image, kernel)
    scipy_ref = correlate2d(test_image, kernel, mode='valid')
    assert np.allclose(ours, scipy_ref), f"Mismatch for {name}"
print("All NumPy convolutions match scipy.signal.correlate2d (valid mode).")

/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_10155/3693071217.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


All NumPy convolutions match scipy.signal.correlate2d (valid mode).


---
**Understanding Convolution as Feature Detection**

**Plain language:** Imagine you have a small stencil with a pattern on it — say, a diagonal line. You slide this stencil across a photograph, and at each position you ask "does the image here look like my stencil?" Where there is a match, you get a strong response; where there is not, you get a weak one. A convolutional layer has many stencils (filters), each looking for a different pattern, and it slides all of them across the entire image. The output is a stack of "heat maps" — one per filter — showing where each pattern was found.

**Building intuition:** A 3x3 filter is a 9-element weight vector. At each spatial position, we extract the 3x3 patch from the image and compute the dot product with the filter. The dot product is maximised when the patch and the filter point in the same direction in 9-dimensional space — i.e., when the image patch looks like the filter. After training, first-layer filters typically converge to edge detectors at various orientations, because edges are the most informative local features in natural images. Deeper layers combine edges into textures (layer 2), textures into parts (layer 3), and parts into objects (layer 4+).

**Formal statement:** A 2D discrete cross-correlation (often called "convolution" in deep learning) between input $\mathbf{I} \in \mathbb{R}^{H \times W}$ and kernel $\mathbf{K} \in \mathbb{R}^{k_H \times k_W}$ is defined as $(\mathbf{I} \star \mathbf{K})[i,j] = \sum_{m=0}^{k_H-1} \sum_{n=0}^{k_W-1} \mathbf{I}[i+m, j+n] \cdot \mathbf{K}[m,n]$. For multi-channel input $\mathbf{I} \in \mathbb{R}^{C_{in} \times H \times W}$ and $C_{out}$ filters $\mathbf{K} \in \mathbb{R}^{C_{out} \times C_{in} \times k_H \times k_W}$, the output feature map $c$ is $\mathbf{O}[c, i, j] = b_c + \sum_{c'=0}^{C_{in}-1} (\mathbf{I}[c'] \star \mathbf{K}[c, c'])[i, j]$. Strictly, deep learning "convolution" is cross-correlation; true convolution would flip the kernel. The distinction is immaterial because the kernel weights are learned (LeCun et al., 1998).

---

---
**Understanding Parameter Sharing and Translation Invariance**

**Plain language:** Imagine you are a security guard looking for a suspicious red bag. You do not need a different pair of eyes for every corner of the room — the same eyes work everywhere. Similarly, a convolutional filter is one set of weights that scans across the entire image. Whether a vertical edge appears in the top-left or bottom-right, the same filter detects it. This is parameter sharing — and it means the network does not need to re-learn "what is an edge" at every spatial location.

**Building intuition:** In an MLP, each hidden neuron has its own weight for every input pixel — there is no weight reuse. A Conv2d layer with a 3x3 filter has only 9 weights (per input channel), but these 9 weights are applied at every spatial position. If the input is 32x32, the filter is applied at 30x30 = 900 positions, but uses the same 9 weights each time. This gives two benefits: (1) far fewer parameters to learn, and (2) translation equivariance — if the input shifts by one pixel, the output feature map shifts by one pixel (the activations move, but do not change in value). Pooling then converts this equivariance into approximate invariance.

**Formal statement:** Let $f_\theta$ denote a convolutional layer parameterised by kernel $\theta \in \mathbb{R}^{k \times k}$. For a translation operator $T_\tau$ that shifts an image by $\tau$ pixels, convolution satisfies $f_\theta(T_\tau(\mathbf{I})) = T_\tau(f_\theta(\mathbf{I}))$ — i.e., the layer is **translation equivariant**. An MLP layer $g_W(\mathbf{x}) = W\mathbf{x} + \mathbf{b}$ with unstructured weight matrix $W$ has no such equivariance. The parameter reduction is dramatic: an MLP connecting a $32 \times 32$ input to $n$ outputs requires $1024n$ weights, while a convolutional layer with $n$ filters of size $k \times k$ requires only $nk^2$ weights, independent of spatial resolution. This is a direct consequence of replacing the unstructured matrix multiplication with a structured (Toeplitz-like) operation (Goodfellow et al., 2016, Ch. 9).

---

## PyTorch Rewrite

Using `nn.Conv2d`, `nn.MaxPool2d`, and `nn.BatchNorm2d` to build a LeNet-style architecture for CIFAR-10.

### Data Loading

CIFAR-10: 60,000 32x32 colour images in 10 classes (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck). 50k train, 10k test.

In [3]:
# CIFAR-10 data loading
transform_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

train_dataset = datasets.CIFAR10('data', train=True, download=True, transform=transform_train)
test_dataset = datasets.CIFAR10('data', train=False, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=0)

# Show some examples (unnormalised)
raw_dataset = datasets.CIFAR10('data', train=True, transform=transforms.ToTensor())
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
classes = raw_dataset.classes
for i, ax in enumerate(axes.flat):
    img, label = raw_dataset[i]
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(classes[label], fontsize=9)
    ax.axis('off')
plt.suptitle('CIFAR-10 Samples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Train: {len(train_dataset):,} images")
print(f"Test:  {len(test_dataset):,} images")
print(f"Classes: {classes}")

  0%|          | 0.00/170M [00:00<?, ?B/s]

  0%|          | 32.8k/170M [00:00<12:54, 220kB/s]

  0%|          | 131k/170M [00:00<04:52, 583kB/s] 

  0%|          | 360k/170M [00:00<02:21, 1.20MB/s]

  0%|          | 786k/170M [00:00<01:15, 2.26MB/s]

...

100%|█████████▉| 170M/170M [00:24<00:00, 8.50MB/s]

100%|██████████| 170M/170M [00:24<00:00, 7.03MB/s]

/Users/rosstaylor/Downloads/Code Repositories/Pj MNEMOSYNE/pj-mnemosyne-repo/Pj-MNEMOSYNE/.venv/lib/python3.14/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Train: 50,000 images
Test:  10,000 images
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_10155/970606990.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
class LeNetCIFAR(nn.Module):
    """
    LeNet-style CNN for CIFAR-10.

    Architecture:
        Conv2d(3, 16, 3, padding=1) -> BN -> ReLU -> MaxPool(2)
        Conv2d(16, 32, 3, padding=1) -> BN -> ReLU -> MaxPool(2)
        Flatten -> FC(32*8*8, 128) -> ReLU -> FC(128, 10)

    Input: (B, 3, 32, 32)
    After conv1 + pool: (B, 16, 16, 16)
    After conv2 + pool: (B, 32, 8, 8)
    Flatten: (B, 2048)
    Output: (B, 10)
    """

    def __init__(self):
        super().__init__()
        # Conv block 1
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.pool = nn.MaxPool2d(2, 2)

        # Conv block 2
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)

        # Fully connected
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # Block 1: conv -> bn -> relu -> pool
        x = self.pool(F.relu(self.bn1(self.conv1(x))))   # (B, 16, 16, 16)
        # Block 2: conv -> bn -> relu -> pool
        x = self.pool(F.relu(self.bn2(self.conv2(x))))   # (B, 32, 8, 8)
        # Flatten and classify
        x = x.view(x.size(0), -1)                         # (B, 2048)
        x = F.relu(self.fc1(x))                            # (B, 128)
        x = self.fc2(x)                                    # (B, 10)
        return x


cnn_model = LeNetCIFAR().to(device)
cnn_params = sum(p.numel() for p in cnn_model.parameters())
print(f"CNN parameters: {cnn_params:,}")
print()
print(cnn_model)
print()

# Verify output shape
dummy = torch.randn(2, 3, 32, 32).to(device)
out = cnn_model(dummy)
print(f"Input shape:  {dummy.shape}")
print(f"Output shape: {out.shape}")
assert out.shape == (2, 10), f"Expected (2, 10), got {out.shape}"

CNN parameters: 268,746

LeNetCIFAR(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc1): Linear(in_features=2048, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)

Input shape:  torch.Size([2, 3, 32, 32])
Output shape: torch.Size([2, 10])


---
**Understanding Receptive Fields and Hierarchical Features**

**Plain language:** When you look at a face, you do not see it as thousands of independent pixels. Your brain processes edges first, then groups edges into eyes, noses, and mouths, then assembles those parts into a face. A CNN does something similar. The first layer sees only a tiny 3x3 patch — enough to detect an edge. The second layer sees a wider region (because it operates on the first layer's output), enough to detect textures and corners. Deeper layers see progressively larger regions, enough to detect whole objects.

**Building intuition:** The receptive field of a neuron is the region of the input image that can influence its value. A 3x3 conv layer has a 3x3 receptive field. Stack two 3x3 layers and the second layer's receptive field is 5x5 — each of its 3x3 patches covers a 3x3 region, and neighbouring patches overlap. Add a 2x2 max pool between them, and the receptive field doubles further. In our LeNet: after conv1 (3x3) the receptive field is 3x3. After pool1 (2x2), it is 4x4. After conv2 (3x3 on the pooled map), each neuron's receptive field covers 8x8 of the original input — a quarter of the CIFAR-10 image. This hierarchical growth is what lets the network build up from low-level to high-level features.

**Formal statement:** For a stack of $L$ convolutional layers with kernel sizes $k_l$ and strides $s_l$, the receptive field $r_L$ of a neuron in layer $L$ with respect to the input is $r_L = 1 + \sum_{l=1}^{L} (k_l - 1) \prod_{j=1}^{l-1} s_j$. For two 3x3 conv layers with stride 1 and a 2x2 max pool (stride 2) after the first: $r_1 = 3$, after pool $r_1' = 4$ (effective), $r_2 = 4 + (3-1) \times 2 = 8$. The hierarchical feature abstraction is formalised by Zeiler & Fergus (2014), who showed via deconvolution that layer 1 learns edges/colours, layer 2 learns textures/corners, layer 3 learns object parts, and layers 4-5 learn whole objects and pose variations.

---

---
**Understanding Pooling (Spatial Downsampling)**

**Plain language:** Imagine you take a photograph and shrink it to a thumbnail. You lose fine details but keep the overall structure — you can still tell it is a cat, even at low resolution. Max pooling does something similar: it looks at a small block of pixels (e.g., 2x2) and keeps only the brightest one, discarding the rest. This makes the representation smaller and makes the network less sensitive to the exact position of features — a cat's ear shifted one pixel to the right still produces the same pooled value.

**Building intuition:** Max pooling with a 2x2 window and stride 2 reduces each spatial dimension by half: a 16x16 feature map becomes 8x8. This has three effects. First, computational savings: the next layer processes 4x fewer spatial positions. Second, it increases the effective receptive field of subsequent layers, because each pooled pixel summarises a 2x2 region. Third, it provides a degree of translation invariance — small shifts in the input (1 pixel) often produce the same pooled output, since the max within each 2x2 window is unchanged. However, max pooling is not differentiable everywhere (gradients flow only through the max element), and some modern architectures replace it with strided convolutions.

**Formal statement:** For input $\mathbf{X} \in \mathbb{R}^{C \times H \times W}$, max pooling with kernel size $k$ and stride $s$ produces $\mathbf{Y} \in \mathbb{R}^{C \times H' \times W'}$ where $H' = \lfloor(H-k)/s\rfloor + 1$ and $\mathbf{Y}[c, i, j] = \max_{0 \le m < k, 0 \le n < k} \mathbf{X}[c, is+m, js+n]$. The subgradient routes through the argmax: $\frac{\partial Y[c,i,j]}{\partial X[c, is+m^*, js+n^*]} = 1$ where $(m^*, n^*) = \text{argmax}_{m,n} X[c, is+m, js+n]$, and 0 elsewhere. Average pooling uses $\mathbf{Y}[c,i,j] = \frac{1}{k^2}\sum_{m,n} \mathbf{X}[c, is+m, js+n]$ and distributes gradients equally. Springenberg et al. (2015) showed that strided convolutions can replace pooling without accuracy loss, but max pooling remains standard in many architectures.

---

---
**Understanding Batch Normalization**

**Plain language:** Imagine a factory assembly line where each station expects parts within a certain size range. If station 1 starts sending parts that are twice as big as expected, station 2 has to constantly re-adjust its tools. Batch normalization prevents this by standardizing the parts between each station — it ensures that the output of each layer has zero mean and unit variance (on average), so the next layer always receives inputs in a predictable range. This makes training faster and more stable.

**Building intuition:** During training, each mini-batch of activations is normalised: $\hat{x}_i = (x_i - \mu_B) / \sqrt{\sigma_B^2 + \epsilon}$, then scaled and shifted by learned parameters $\gamma$ and $\beta$. This does two things: (1) it reduces "internal covariate shift" — the distribution of layer inputs changes less as earlier layers update, so later layers do not need to constantly re-adapt; (2) it smooths the loss landscape (Santurkar et al., 2018 showed this is the primary mechanism, not covariate shift reduction). For CNNs, BatchNorm2d normalises across the batch and spatial dimensions but independently per channel — each channel gets its own $\gamma$ and $\beta$. At test time, running averages of $\mu$ and $\sigma^2$ replace per-batch statistics.

**Formal statement:** Given a mini-batch $\mathcal{B} = \{x_1, \ldots, x_m\}$, Batch Normalization (Ioffe & Szegedy, 2015) computes: $\mu_\mathcal{B} = \frac{1}{m}\sum_{i=1}^{m} x_i$, $\sigma_\mathcal{B}^2 = \frac{1}{m}\sum_{i=1}^{m}(x_i - \mu_\mathcal{B})^2$, $\hat{x}_i = \frac{x_i - \mu_\mathcal{B}}{\sqrt{\sigma_\mathcal{B}^2 + \epsilon}}$, $y_i = \gamma \hat{x}_i + \beta$. For convolutional layers with feature maps $\mathbf{X} \in \mathbb{R}^{B \times C \times H \times W}$, the statistics are computed per-channel across the $(B, H, W)$ dimensions: $\mu_c = \frac{1}{BHW}\sum_{b,i,j} X_{b,c,i,j}$. The learnable affine parameters $\gamma_c, \beta_c \in \mathbb{R}$ allow the network to undo the normalisation if beneficial. During inference, exponential moving averages of $\mu$ and $\sigma^2$ from training are used, making the transform deterministic.

---

## Training Run

20 epochs on CIFAR-10 with SGD + momentum. We track train loss, train accuracy, and test accuracy per epoch.

In [5]:
def train_epoch(model, loader, optimizer, criterion, device):
    """Train for one epoch, return (avg_loss, accuracy)."""
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)
        correct += (logits.argmax(1) == y_batch).sum().item()
        total += X_batch.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def eval_model(model, loader, device):
    """Evaluate model, return accuracy."""
    model.eval()
    correct, total = 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        logits = model(X_batch)
        correct += (logits.argmax(1) == y_batch).sum().item()
        total += X_batch.size(0)
    return correct / total


# Train CNN
torch.manual_seed(42)
cnn_model = LeNetCIFAR().to(device)
optimizer = torch.optim.SGD(cnn_model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

cnn_train_losses, cnn_train_accs, cnn_test_accs = [], [], []
n_epochs = 20

t0 = time.time()
for epoch in range(1, n_epochs + 1):
    train_loss, train_acc = train_epoch(cnn_model, train_loader, optimizer, criterion, device)
    test_acc = eval_model(cnn_model, test_loader, device)

    cnn_train_losses.append(train_loss)
    cnn_train_accs.append(train_acc)
    cnn_test_accs.append(test_acc)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:2d}/{n_epochs}  loss={train_loss:.4f}  "
              f"train_acc={train_acc:.4f}  test_acc={test_acc:.4f}")

elapsed = time.time() - t0
print(f"\nTraining complete in {elapsed:.1f}s")
print(f"Final test accuracy: {cnn_test_accs[-1]:.4f}")

# Silent correctness assertion
assert cnn_test_accs[-1] > 0.60, (
    f"CNN test accuracy {cnn_test_accs[-1]:.4f} below 0.60 threshold"
)

Epoch  1/20  loss=1.3442  train_acc=0.5120  test_acc=0.5865


Epoch  5/20  loss=0.6881  train_acc=0.7569  test_acc=0.6994


Epoch 10/20  loss=0.3927  train_acc=0.8628  test_acc=0.7111


Epoch 15/20  loss=0.1887  train_acc=0.9366  test_acc=0.7067


Epoch 20/20  loss=0.0758  train_acc=0.9772  test_acc=0.7248

Training complete in 208.9s
Final test accuracy: 0.7248


In [6]:
# Plot learning curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, n_epochs + 1), cnn_train_losses, 'b-o', markersize=3, label='Train loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('CNN Training Loss (CIFAR-10)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, n_epochs + 1), cnn_train_accs, 'b-o', markersize=3, label='Train acc')
axes[1].plot(range(1, n_epochs + 1), cnn_test_accs, 'r-s', markersize=3, label='Test acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('CNN Train vs Test Accuracy (CIFAR-10)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0.3, 0.85)

plt.tight_layout()
plt.savefig('../assets/cnn_learning_curves.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("Saved: ../assets/cnn_learning_curves.png")

Saved: ../assets/cnn_learning_curves.png


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_10155/4264532249.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Internal Probing

Visualise the learned first-layer convolutional filters. In a trained CNN, these should resemble edge detectors at various orientations and colour channels — this is a universal finding across CNN architectures (Zeiler & Fergus, 2014; Krizhevsky et al., 2012).

In [7]:
# Visualise first-layer filters (conv1: 16 filters of shape 3x3x3)
filters = cnn_model.conv1.weight.detach().cpu().numpy()  # (16, 3, 3, 3)
print(f"conv1 filter shape: {filters.shape}")

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    # Normalise each filter to [0, 1] for display
    f = filters[i].transpose(1, 2, 0)  # (3, 3, 3) -> (3, 3, 3) HWC
    f_min, f_max = f.min(), f.max()
    f_norm = (f - f_min) / (f_max - f_min + 1e-8)
    ax.imshow(f_norm, interpolation='nearest')
    ax.set_title(f'Filter {i}', fontsize=9)
    ax.axis('off')

plt.suptitle('Learned First-Layer Filters (conv1)\n'
             'Should show edge/colour detectors',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../assets/cnn_filters.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("Saved: ../assets/cnn_filters.png")

conv1 filter shape: (16, 3, 3, 3)


Saved: ../assets/cnn_filters.png


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_10155/3334480624.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Show what the filters "see" — apply conv1 to a sample image and display feature maps
cnn_model.eval()
sample_img, sample_label = test_dataset[0]
sample_img_device = sample_img.unsqueeze(0).to(device)

with torch.no_grad():
    conv1_out = F.relu(cnn_model.bn1(cnn_model.conv1(sample_img_device)))  # (1, 16, 32, 32)

feature_maps = conv1_out.squeeze(0).cpu().numpy()  # (16, 32, 32)

fig, axes = plt.subplots(2, 9, figsize=(18, 4.5))
# Show original image (unnormalised) in first position
raw_img = datasets.CIFAR10('data', train=False, transform=transforms.ToTensor())[0][0]
axes[0, 0].imshow(raw_img.permute(1, 2, 0).numpy())
axes[0, 0].set_title(f'Input\n({classes[sample_label]})', fontsize=9)
axes[0, 0].axis('off')

# Show 16 feature maps
for i in range(16):
    row = (i + 1) // 9
    col = (i + 1) % 9
    axes[row, col].imshow(feature_maps[i], cmap='viridis')
    axes[row, col].set_title(f'FM {i}', fontsize=8)
    axes[row, col].axis('off')

# Hide last unused subplot
axes[1, 8].axis('off')

plt.suptitle('Feature Maps After conv1 + BN + ReLU', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_10155/1214956736.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Comparison: MLP on the Same Data

To demonstrate that CNNs win on image tasks, we train a simple MLP (flatten + fully connected layers) on the same CIFAR-10 data for the same number of epochs. The MLP destroys spatial structure by flattening — it cannot exploit local patterns, edges, or textures.

In [9]:
class CIFAR_MLP(nn.Module):
    """MLP baseline for CIFAR-10: flatten -> 512 -> 256 -> 10."""

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3 * 32 * 32, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)      # flatten: (B, 3072)
        x = F.relu(self.fc1(x))         # (B, 512)
        x = F.relu(self.fc2(x))         # (B, 256)
        x = self.fc3(x)                 # (B, 10)
        return x


torch.manual_seed(42)
mlp_model = CIFAR_MLP().to(device)
mlp_params = sum(p.numel() for p in mlp_model.parameters())

print(f"CNN parameters:  {cnn_params:,}")
print(f"MLP parameters:  {mlp_params:,}")
print(f"MLP has {mlp_params / cnn_params:.1f}x more parameters than CNN")
print()

mlp_optimizer = torch.optim.SGD(mlp_model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)

mlp_train_losses, mlp_train_accs, mlp_test_accs = [], [], []

t0 = time.time()
for epoch in range(1, n_epochs + 1):
    train_loss, train_acc = train_epoch(mlp_model, train_loader, mlp_optimizer, criterion, device)
    test_acc = eval_model(mlp_model, test_loader, device)

    mlp_train_losses.append(train_loss)
    mlp_train_accs.append(train_acc)
    mlp_test_accs.append(test_acc)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:2d}/{n_epochs}  loss={train_loss:.4f}  "
              f"train_acc={train_acc:.4f}  test_acc={test_acc:.4f}")

elapsed = time.time() - t0
print(f"\nMLP training complete in {elapsed:.1f}s")
print(f"MLP final test accuracy: {mlp_test_accs[-1]:.4f}")
print(f"CNN final test accuracy: {cnn_test_accs[-1]:.4f}")
print(f"CNN advantage: +{cnn_test_accs[-1] - mlp_test_accs[-1]:.4f}")

# CNN should beat MLP
assert cnn_test_accs[-1] > mlp_test_accs[-1], "CNN should outperform MLP on CIFAR-10"

CNN parameters:  268,746
MLP parameters:  1,707,274
MLP has 6.4x more parameters than CNN



Epoch  1/20  loss=1.6961  train_acc=0.3981  test_acc=0.4638


Epoch  5/20  loss=1.1524  train_acc=0.5955  test_acc=0.5285


Epoch 10/20  loss=0.8320  train_acc=0.7063  test_acc=0.5504


Epoch 15/20  loss=0.5672  train_acc=0.8003  test_acc=0.5438


Epoch 20/20  loss=0.3714  train_acc=0.8690  test_acc=0.5387

MLP training complete in 60.2s
MLP final test accuracy: 0.5387
CNN final test accuracy: 0.7248
CNN advantage: +0.1861


In [10]:
# CNN vs MLP comparison plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Test accuracy comparison
axes[0].plot(range(1, n_epochs + 1), cnn_test_accs, 'b-o', markersize=3,
             label=f'CNN ({cnn_params:,} params)', linewidth=2)
axes[0].plot(range(1, n_epochs + 1), mlp_test_accs, 'r-s', markersize=3,
             label=f'MLP ({mlp_params:,} params)', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Test Accuracy', fontsize=12)
axes[0].set_title('CNN vs MLP: Test Accuracy on CIFAR-10', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Training loss comparison
axes[1].plot(range(1, n_epochs + 1), cnn_train_losses, 'b-o', markersize=3,
             label='CNN', linewidth=2)
axes[1].plot(range(1, n_epochs + 1), mlp_train_losses, 'r-s', markersize=3,
             label='MLP', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Training Loss', fontsize=12)
axes[1].set_title('CNN vs MLP: Training Loss', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../assets/cnn_vs_mlp.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("Saved: ../assets/cnn_vs_mlp.png")

Saved: ../assets/cnn_vs_mlp.png


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_10155/2177120692.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
**Understanding Why CNNs Beat MLPs on Images**

**Plain language:** Imagine you are trying to find Wally in a "Where's Wally?" book. An MLP approach would be like memorising the exact position of every red-and-white-striped pixel for every page you have seen — it works if Wally is always in the same spot, but fails completely when he moves. A CNN approach is like learning what Wally *looks like* (red stripy shirt, bobble hat) and then scanning across the page for that pattern. The CNN's approach generalises, because the pattern detector works wherever Wally happens to be.

**Building intuition:** The CNN has three structural advantages. First, **locality**: a 3x3 filter can only connect nearby pixels, encoding the prior that spatial neighbours are more informative than distant pixels. An MLP connecting all 3,072 inputs to each hidden neuron wastes capacity learning that pixel (0, 0) is unrelated to pixel (31, 31). Second, **parameter sharing**: our CNN uses only ~270k parameters to outperform an MLP with ~1.7M parameters, because the same 3x3 filters are reused at every spatial position. Third, **hierarchical composition**: pooling + deeper layers let the CNN build edge -> texture -> part -> object representations, each layer operating at a coarser spatial scale. The MLP has no such hierarchy — it must learn everything in a single flat representation.

**Formal statement:** The inductive bias of a CNN is formalised as the group equivariance property. Under the translation group $(\mathbb{Z}^2, +)$, a convolutional layer satisfies $f_\theta \circ T_\tau = T_\tau \circ f_\theta$ for any translation $\tau$. This equivariance means the network need not learn separate detectors for each spatial location — a single kernel generalises across all positions. Cohen & Welling (2016) generalised this to arbitrary groups (rotations, reflections) with Group Equivariant CNNs. From a sample complexity perspective, the parameter sharing reduces the effective number of free parameters from $O(d \cdot n)$ (MLP) to $O(k^2 \cdot n)$ (CNN), where $d$ is input dimensionality, $n$ is the number of features, and $k$ is kernel size. For CIFAR-10 ($d=3072$, $k=3$), this is a $\sim340\times$ reduction per layer, yielding better generalisation with less data (Neyshabur et al., 2017).

---

## Structured Interpretation

### Results
- **NumPy convolution from scratch** matches `scipy.signal.correlate2d` exactly, confirming correct implementation of the sliding dot product operation
- **LeNet-style CNN** (2 conv + BN + pool + 2 FC layers): test accuracy > 60% on CIFAR-10 after 20 epochs with SGD + momentum
- **MLP baseline** (flatten + 512 + 256 + 10): underperforms CNN despite having significantly more parameters
- **First-layer filters** show edge-like and colour-selective patterns, consistent with the universal finding that early conv layers learn Gabor-like features
- **Feature maps** show spatially structured activation patterns, confirming that conv layers preserve and exploit spatial information

### Findings
- The CNN achieves higher accuracy than the MLP with fewer parameters, demonstrating that inductive bias (locality, parameter sharing) is more valuable than raw capacity for image tasks
- Batch normalisation enables stable training without careful learning rate tuning — the BN layers keep activations in a healthy range throughout training
- First-layer filter visualisation confirms the model learns meaningful low-level features (edges at various orientations, colour contrasts) rather than memorising noise
- The accuracy gap between CNN and MLP would widen further with more epochs, data augmentation, or deeper architectures

### Implications
- For any task with spatial structure (images, grids, spatially-arranged sensor data), CNNs should be the default starting point, not MLPs
- The principle of matching architecture to data structure generalises: RNNs for sequences, Transformers for sets with attention, Graph NNs for graph-structured data
- For the MNEMOSYNE project, if spatial features (e.g., geographic proximity of MEDEVAC assets) are relevant, convolutional layers could encode that structure

### Considerations
- Our LeNet-style model is deliberately small for pedagogical clarity — modern CNNs (ResNet, EfficientNet) use residual connections, deeper stacks, and data augmentation to reach 95%+ on CIFAR-10
- We used no data augmentation (random crops, flips) which would significantly improve generalisation
- The comparison is intentionally unfair to the MLP — the point is to show that architecture matters, not to exhaustively tune both models
- Batch normalisation introduces a train/eval mode distinction that can cause subtle bugs if forgotten (always call `model.eval()` before inference)